In [1]:
!pip install -q -U transformers accelerate bitsandbytes

In [2]:
pip install chainlit

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [ ]:
!pip install -q \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    pypdf

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)
MODEL_NAME = "Qwen/Qwen3-4B"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/RAG_chatbot/YOLOv10_Tutorials.pdf"

loader = PyPDFLoader(FILE_PATH)

documents = loader.load()

print("Number of pages:", len(documents))

In [ ]:
print(documents[0].page_content[:1000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
question = "what is the author of the document?"

relevant_docs = retriever.invoke(question)

for i, doc in enumerate(relevant_docs):
    print(f"\n--- Chunk {i + 1} ---")
    print(doc.page_content[:500])

In [ ]:
messages = []

while True:
    question = input("You: ")

    if question.lower() in ["exit", "quit", "q"]:
        print("Chat ended.")
        break

    # =========================
    # 1. RETRIEVAL
    # =========================

    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in relevant_docs
    )

    # =========================
    # 2. TẠO RAG PROMPT
    # =========================

    rag_prompt = f"""
Use the following context to answer the question.

If the answer cannot be found in the context,
say that you do not know based on the provided document.

Context:
{context}

Question:
{question}
"""

    # =========================
    # 3. LƯU USER MESSAGE
    # =========================

    messages.append({
        "role": "user",
        "content": rag_prompt
    })

    # =========================
    # 4. CHAT TEMPLATE
    # =========================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    # =========================
    # 5. GENERATION
    # =========================

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.8
    )

    output_ids = generated_ids[0][
        model_inputs.input_ids.shape[1]:
    ]

    answer = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )

    print("\nQwen:", answer)
    print()

    # =========================
    # 6. CHAT HISTORY
    # =========================

    messages.append({
        "role": "assistant",
        "content": answer
    })